# 模型

In [ ]:
import torch

class Attention(torch.nn.Module):
    def __init__(self,d_model,num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.w_q = torch.nn.Parameter(
            torch.randn(d_model, d_model) * d_model**-0.5
        )
        self.w_k = torch.nn.Parameter(
            torch.randn(d_model, d_model) * d_model**-0.5
        )
        self.w_v = torch.nn.Parameter(
            torch.randn(d_model, d_model) * d_model**-0.5
        )

        self.out_proj = torch.nn.Linear(d_model, d_model) #线性层
        self.norm = torch.nn.LayerNorm(d_model) #归一和线性层
        self.norm2 = torch.nn.LayerNorm(d_model)

        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(d_model, 4 * d_model),
            torch.nn.GELU(),
            torch.nn.Linear(4 * d_model, d_model)
        )#mlp 层

    def forward(self, x, prefix_len, remove_distance=None):
        original = x
        x = self.norm(x) #[4,80]
        q = x@self.w_q #[4*80*1024]
        k = x@self.w_k
        v = x@self.w_v

# 加上multihead 维度
        B, T, D = x.shape

        q = q.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2) #[4,8,80,128]
        k = k.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        score = q @ k.transpose(-2,-1) / q.shape[-1] ** 0.5 #[4*8*80*80]最后是相关性矩阵
        # AR mask
        allowed = torch.ones(q.shape[-2],q.shape[-2],dtype=torch.bool,device=x.device).tril()
        # BD mask
        allowed[prefix_len:, :] = True # 最后几个token 全部可以看 [80*80]

        if remove_distance is not None: #第一个生成位置 看不到 remove_distance 前的内容
            assert 1 <= remove_distance <= prefix_len
            allowed[prefix_len, prefix_len - remove_distance] = False

        scores = score.masked_fill(~allowed, float("-inf"))
        weights = scores.softmax(dim=-1)
        context = weights @ v # [4,8,80，128]每个位置的表示混合
#把multihead 重新合成到single head
        context = context.transpose(1,2).contiguous().reshape(B,T,D)

        #记录当前的attention
        self.record = {
            "q": q.detach().clone(), # [batch * head * sequence * head d model]
            "attention": weights.detach().clone(), # [batch * head  * sequence * sequence]
            "context": context.detach().clone(), # [batch * head * sequence  * head d model]
        }
        


        output = self.out_proj(context)
        h = output + original
        h = h + self.mlp(self.norm2(h))
        return h

    
class ToyBD(torch.nn.Module):
    def __init__(self,vocab_size,d_model,max_seq_len,num_heads,num_layer):
        super().__init__()
        self.token_embedding = torch.nn.Embedding(vocab_size,d_model)
        self.position_embedding = torch.nn.Embedding(max_seq_len,d_model)

# 这里是变成了 multi——layer
        self.layers = torch.nn.ModuleList([Attention(d_model=d_model,num_heads=num_heads) for _ in range(num_layer)])
        self.final_norm = torch.nn.LayerNorm(d_model)
        self.output_head = torch.nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, prefix_len,remove_distance=None):
        seq_len = input_ids.shape[1] #[batchsize,len]
        positions = torch.arange(seq_len, device=input_ids.device)

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )#[batchsize,len,d_model]

        h = x
        # 这里增加层数
        for layer in self.layers:
            h = layer(h,prefix_len, remove_distance = remove_distance)

        logits = self.output_head(self.final_norm(h))#[batchsize,len,vocab_size]

        return logits


# 数据

In [ ]:
torch.manual_seed(0)
from pathlib import Path
data_path = Path.cwd() / 'a2a_shakesphere.txt'
text = data_path.read_text(encoding= 'utf-8').lower()
chars = sorted(set(text))

chars_to_ids = {c:i for i,c in enumerate(chars)}
ids_to_chars = {i:c for i,c in enumerate(chars)}

mask_id = len(chars)
vocab_size = len(chars)+1

data = torch.tensor([chars_to_ids[c] for c in text],dtype=torch.long)
split = int(len(data)*0.9)
train_data = data[:split]
test_data = data[split:]


def get_batch(data,prefix_len,block_size,batch_size):

    seq_len = prefix_len+block_size

    starts = torch.randint(0,len(data)-seq_len+1,(batch_size,))


    sample = torch.stack(
        [data[start.item():start.item()+seq_len] for start in starts]
        )

    input_ids = sample.clone()

    # 部分遮挡 50%
    mask_positions = torch.rand(batch_size, block_size) < 0.5
    mask_positions[~mask_positions.any(dim=1), 0] = True # 至少把第一个设置成 True
    input_ids[:,prefix_len:]= input_ids[:,prefix_len:].masked_fill(mask_positions,mask_id)

    # 25% 几率全遮挡
    full_mask_rows = torch.rand(batch_size)< 0.25
    input_ids[full_mask_rows,prefix_len:] = mask_id
    mask_positions[full_mask_rows, :] = True

    target = sample[:,prefix_len:].clone()
    return sample, input_ids, target,mask_positions  # batch * sequence len

prefix_len,block_size,batch_size = 64,16,4
sample,input_ids,target,mask_positions = get_batch(train_data,64,16,4)

print(input_ids)

# 训练

In [ ]:
d_model = 128
prefix_len = 64
max_seq_len = 512
block_size = 4
print(block_size)

num_steps = 500
num_heads = 4
num_layers = 2

batch_size = 64
torch.set_num_threads(2)


bd_model = ToyBD(vocab_size,d_model=d_model, max_seq_len=max_seq_len,num_heads=num_heads,num_layer=num_layers)
optimizer = torch.optim.AdamW(bd_model.parameters(),lr = 1e-3,weight_decay= 0.01)

loss_record = []
for i in range(num_steps):
    sample ,input_ids, targets,mask_positions = get_batch(train_data,prefix_len,block_size,batch_size)
    optimizer.zero_grad()
    logits = bd_model.forward(input_ids=input_ids,prefix_len=prefix_len)
    block_logits = logits[:,prefix_len:,:]
    loss = torch.nn.functional.cross_entropy(block_logits[mask_positions],targets[mask_positions]) # [4, 16, vocablen] and # [4, 16] reshape: 自动合并成目标维度
    # 取出后自动变成一维
    loss.backward()
    loss_record.append(loss.item())
    print('step',i,'loss',loss.item())
    optimizer.step()
 
print(loss_record)
import matplotlib.pyplot as plt
plt.plot(loss_record)

# 一次rollout

In [ ]:
forward_time = 3

bd_model.eval()  # 切换到推理模式

# 从验证集取64个字符作为开头，不输入后面的真实答案
generated = test_data[1000:1000+prefix_len].clone()
device = next(bd_model.parameters()).device
generated = generated.to(device)

print("前文：")
print("".join(ids_to_chars[i] for i in generated.tolist()))

with torch.no_grad():  # 推理时不计算梯度
    for _ in range(forward_time):  # 生成5块，每块16个字符
        # 始终只保留最近64个字符作为模型的上下文
        prefix = generated[-prefix_len:]

        masks = torch.full(
            (block_size,), mask_id,
            dtype=torch.long, device=device
        )

        # [80] → [1, 80]，增加batch维度
        inputs = torch.cat([prefix, masks]).unsqueeze(0)

        for denoise_step in range(block_size):
            logits = bd_model.forward(inputs,prefix_len)
            block_logits = logits[0,prefix_len:].clone()
            block_logits[:, mask_id] = float('-inf')
            prob = block_logits.softmax(dim = -1)

            confidence, predicted_ids = prob.max(dim=-1)

            # 已填好的位置不再参与选择
            unknown = inputs[0, prefix_len:] == mask_id
            confidence = confidence.masked_fill(~unknown, -1.0)

            position = confidence.argmax().item()
            inputs[0, prefix_len+position] = predicted_ids[position]

        generated = torch.cat([generated, inputs[0,prefix_len:]])


record = bd_model.layers[0].record #最后一次forward 产生的最后一个词 的每个layer（此时只有自己是masked 前后都被生成出来了）的每个head

print(record['attention'])

print("\n完整结果：")
print("".join(ids_to_chars[i] for i in generated.tolist()))

# Agent result 复现 先看attention （这是最后一组生成最后一个token时的attention）

In [ ]:
import torch
def plot_matrices(
    matrices, prefix_len,
    title="", cmap="coolwarm", vmin=0, vmax=1,
    colorbar_label="", mark_diagonal=False
):
    import matplotlib.pyplot as plt
    matrices = matrices.detach().cpu()
    n_layers, n_heads, T, S = matrices.shape

    fig, axes = plt.subplots(
        n_layers, n_heads,
        figsize=(3 * n_heads, 3 * n_layers),
        squeeze=False,
        layout="constrained"
    )

    for layer_id in range(n_layers):
        for head_id in range(n_heads):
            ax = axes[layer_id, head_id]

            im = ax.imshow(
                matrices[layer_id, head_id].numpy(),
                cmap=cmap, vmin=vmin, vmax=vmax,
                origin="upper",
                interpolation="nearest",
                aspect="auto"
            )

            if mark_diagonal and T == S:
                ax.plot(
                    range(T), range(T),
                    color="white", linestyle=":", linewidth=0.8
                )

            ax.axvline(prefix_len - 0.5, color="black", linewidth=0.6)
            if T == S:
                ax.axhline(prefix_len - 0.5, color="black", linewidth=0.6)

            if layer_id == 0:
                ax.set_title(f"Head {head_id}")
            if head_id == 0:
                ax.set_ylabel(f"Layer {layer_id}\nRow position")
            if layer_id == n_layers - 1:
                ax.set_xlabel("Column position")

    fig.suptitle(title)
    fig.colorbar(
        im, ax=axes.ravel().tolist(),
        label=colorbar_label, shrink=0.6
    )
    plt.show()
    return fig, axes


attention_matrices = torch.stack([
    layer.record["attention"][0]
    for layer in bd_model.layers
])
# [layer, head, query位置, key位置]

plot_matrices(
    attention_matrices,
    prefix_len,
    title="last_masked_token_attention_graph",
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    colorbar_label="Attention weight",
    mark_diagonal=True
)


# 再看生成第一个mask 时候q 和前面的q 有多相似

In [ ]:
bd_model.eval()
device = next(bd_model.parameters()).device

ahead = -15
after = 5

# 随机采一段样本
prefix = test_data[1000:1000 +prefix_len].to(device)
masks = torch.full(
    (block_size,), mask_id,
    dtype=torch.long, device=device
)
probe_ids = torch.cat([prefix, masks]).unsqueeze(0)
with torch.no_grad():
    probe_logits = bd_model.forward(probe_ids,prefix_len)

print(probe_ids.shape)



record = bd_model.layers[1].record  # 第一层  第一个sequence 第一个head
q = record["q"]
q_mask = q[0, 0, prefix_len, :] # 第一个mask 的query
q_clean = q[0, 0, prefix_len - 1, :] #前一个mask 的query
print('当前的query:', q_mask)


for distance in range(1, abs(ahead)):
    q_clean = q[0, 0, prefix_len - distance, :]

    similarity = torch.nn.functional.cosine_similarity(
        q_mask, q_clean, dim=0
    )

    print("前", distance, "个位置：", similarity.item())


q = torch.stack([
    layer.record['q'][0]
    for layer in bd_model.layers
])


start = max(0, prefix_len + ahead)
end = min(q.shape[-2], prefix_len + after)

q_local = q[:, :, start:end, :]
print(q_local.shape)  # [2, 4, 19, 32]

q_unit = torch.nn.functional.normalize(q_local, dim=-1)
qq_similarity = q_unit @ q_unit.transpose(-2, -1)
row = prefix_len - start  # 第一个MASK在局部矩阵里的位置：15
qq_row = qq_similarity[:, :, row:row + 1, :]

plot_matrices(
    qq_row,
    prefix_len=prefix_len - start,  # 64 - 49 = 15
    title=f"Q cosine similarity: positions {start}–{end - 1}",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    colorbar_label="Q cosine similarity"
)


# 所有q 减去均值之后的结果

In [ ]:
# 每层、每个head分别计算前16个clean Q的均值
mean_q = q[:, :, prefix_len - 16:prefix_len, :].mean(
    dim=-2, keepdim=True
)
# [2, 4, 1, 32]

q_centered = q - mean_q
# [2, 4, 68, 32]

# 截取与原图相同的位置，再算余弦相似度
q_local_centered = q_centered[:, :, start:end, :]
q_unit = torch.nn.functional.normalize(q_local_centered, dim=-1)
qq_centered = q_unit @ q_unit.transpose(-2, -1)

row = prefix_len - start

plot_matrices(
    qq_centered[:, :, row:row + 1, :],
    prefix_len=prefix_len - start,
    title="Centered Q cosine similarity",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    colorbar_label="Centered Q cosine similarity"
)

# 接下来看h 的相似度

In [ ]:
context = torch.stack([layer.record['context'][0] for layer in bd_model.layers])
L, T, D = context.shape
H = bd_model.layers[0].num_heads
context = context.reshape(L, T, H, D // H).transpose(1, 2)

context_local = context[:, :, start:end, :]

context_unit = torch.nn.functional.normalize(
    context_local, dim=-1
)

cc_similarity = context_unit @ context_unit.transpose(-2, -1)
# [2, 4, 19, 19]

row = prefix_len - start
 # 专门把正在产生词的这个token扣出来
plot_matrices(
    cc_similarity[:, :, row:row + 1, :],
    prefix_len=row,
    title="First MASK context cosine similarity",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    colorbar_label="Context cosine similarity"
)


# 最后做移除连接实验 生成p 位置的mask 的时候看不到前一位的信息

In [ ]:
baseline_ids = probe_ids.detach().clone()

bd_model.eval()
with torch.no_grad():
    baseline_logits = bd_model(baseline_ids, prefix_len).detach().clone()

# 第一个MASK位置的词表概率
baseline_probs = baseline_logits[0, prefix_len, :].softmax(dim=-1)

best_id = baseline_probs.argmax().item()

print(baseline_probs.shape)  # [40]
print("原预测：", ids_to_chars.get(best_id, "[MASK]"))
print("原预测概率：", baseline_probs[best_id].item())

with torch.no_grad():
    removed_logits = bd_model(
        baseline_ids, prefix_len,
        remove_distance=1
    )

removed_probs = removed_logits[0, prefix_len, :].softmax(dim=-1)
before_id = baseline_probs.argmax().item()
after_id = removed_probs.argmax().item()

print("干预前：", repr(ids_to_chars.get(before_id, "[MASK]")))
print("干预后：", repr(ids_to_chars.get(after_id, "[MASK]")))
print("top1是否改变：", before_id != after_id)

print("原预测字符的概率：")
print("干预前：", baseline_probs[before_id].item())
print("干预后：", removed_probs[before_id].item())